In [36]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, convert_to_messages
from langchain_chroma import Chroma
import gradio as gr

import logging

In [12]:
load_dotenv(override=True)

True

In [29]:
logger = logging.getLogger(__name__)

In [ ]:
# Define the Chroma DB Name
DB_NAME = "vector_db"

# Create the embedding and chroma ref
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

chroma = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4617.60it/s]


In [ ]:
# Create the retriever and LLM abstractions - called runnables

retriever = chroma.as_retriever()
llm = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.5-flash-lite")

In [14]:
# Testing retriever

retriever.invoke("Who is Avery?")

[Document(id='54fcfa30-9b66-4ac7-a034-d7efcd69029b', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [52]:
SYSTEM_PROMPT_PREFIX = """
You are a knowledge assistant for an insurance company named insurellm who responds to user queries. 
You might be given additional context regarding the user query. However, it might be possible that you don't receive a context at times.
Your response should be crisp, clear, short and direct.
Do not respond to questions with hallucinations if you do not have the context. Gracefully say so. 
If you have the context, respond. You should also be able to handle typos really well.

The context is as so - \n\n
"""

In [53]:
def combine_user_messages(message, history):
    prior_messages = "\n".join(m["content"][0]["text"] for m in history if m["role"] == "user")
    return prior_messages + "\n" + message

In [54]:
def retrieve_and_answer(message, history):
    combined_message = combine_user_messages(message, history)
    relevant_documents = retriever.invoke(combined_message, k=5)
    system_prompt = SYSTEM_PROMPT_PREFIX + "\n\n".join(doc.page_content for doc in relevant_documents)
    messages = [SystemMessage(system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(message))
    response = llm.invoke(messages)
    return response.content

In [55]:
gr.ChatInterface(fn=retrieve_and_answer).launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


c:\Users\LENOVO\Documents\projects\llm_mini_projects\insurellm-proper-rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\LENOVO\Documents\projects\llm_mini_projects\insurellm-proper-rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\LENOVO\Documents\projects\llm_mini_projects\insurellm-proper-rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
